In [8]:
camera_positions

[array([[ 1.0000000e+00, -4.4465920e-05,  1.2730720e-04, -3.3908340e-04],
        [ 4.4460499e-05,  1.0000000e+00,  4.2564108e-05,  5.8432022e-04],
        [-1.2730909e-04, -4.2558448e-05,  1.0000000e+00,  3.2929056e-03],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00]],
       dtype=float32),
 array([[ 1.00000000e+00, -4.27119339e-05,  1.09070017e-04,
         -1.88171267e-04],
        [ 4.27068917e-05,  1.00000000e+00,  4.62108183e-05,
          3.40934261e-04],
        [-1.09071996e-04, -4.62061616e-05,  1.00000000e+00,
          1.99494394e-03],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]], dtype=float32),
 array([[ 1.0000000e+00, -4.1602289e-05,  9.5438976e-05, -1.3249334e-04],
        [ 4.1597465e-05,  1.0000000e+00,  5.0517701e-05,  2.6603258e-04],
        [-9.5441072e-05, -5.0513729e-05,  1.0000000e+00,  1.6464900e-03],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00]],
       dtype=float

In [4]:
o3d.visualization.draw_geometries(geometries)

In [2]:
o3d.visualization.draw_geometries(geometries_global)

In [2]:
o3d.visualization.draw_geometries(global_geoms)

In [16]:
o3d.visualization.draw_geometries(geometries1)

In [30]:
o3d.visualization.draw_geometries(geometries1)

In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_5.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

# images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"]
# ...

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    
    

]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions1 = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries = []
camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    points_cam = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_cam)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries.append(pcd1)
    geometries.append(camera_frame)


    # PLY += pcd1

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


PointCloud with 1015280 points.

In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_5_filtered.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
]

views = load_images(images)

# Run inference
predictions1 = model.infer(
    views,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

geometries = []
camera_positions = []

# --- SET YOUR FILTER DISTANCE HERE ---
# This will remove points farther than this distance (in meters/units)
# from the camera's origin.
MAX_FILTER_DISTANCE = 30.0 
print(f"Filtering points farther than {MAX_FILTER_DISTANCE} units from their camera.")
# ----------------------------------------

# PLY = o3d.geometry.PointCloud() # Use this for saving a combined cloud

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    # Get raw points, colors, and the pose
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    # --- START: New Filtering Logic ---
    
    # 1. Get the camera's origin (its position in the world frame)
    camera_origin = camera_pose[:3, 3]
    
    # 2. Calculate the distance of each point from the camera origin
    distances = np.linalg.norm(points_world - camera_origin, axis=1)
    
    # 3. Create a boolean mask for points *within* the distance
    mask = distances <= MAX_FILTER_DISTANCE
    
    # 4. Apply the mask to the points and colors
    filtered_points = points_world[mask]
    filtered_colors = colors[mask]
    
    # --- END: New Filtering Logic ---

    # 5. Create the point cloud from the *filtered* data
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(filtered_points)
    pcd1.colors = o3d.utility.Vector3dVector(filtered_colors)
    
    # Get camera center for drawing the path
    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    # Create the camera coordinate frame visualization
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    # Add the filtered cloud and camera frame
    geometries.append(pcd1)
    geometries.append(camera_frame)

    # PLY += pcd1 # Add to the combined cloud for saving

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()

# Draw the camera path
if len(camera_positions) > 1:
    line_points = o3d.utility.Vector3dVector(camera_positions)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    camera_path.paint_uniform_color([1, 0, 0]) # Red path
    
    geometries.append(camera_path)

# Apply the final coordinate transform
transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)

# 4. Display all the geometries together in one window
print(f"Displaying combined scene with {len(predictions1)} filtered point clouds...")
o3d.visualization.draw_geometries(geometries)

# Save the combined PLY file
# print(f"Saving to {output_filename}...")
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Filtering points farther than 30.0 units from their camera.
Displaying combined scene with 5 filtered point clouds...


In [ ]:
o3d.visualization.draw_geometries(geometries)

In [16]:
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    

]

views2 = load_images(batch2)

# Run inference (this will process all images in the list)
preds2 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

last_pose2 = None

PLY = o3d.geometry.PointCloud()


for i, pred in enumerate(preds2):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose2 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [14]:
o3d.visualization.draw_geometries(geometries)

In [17]:
batch3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
    

]

views3 = load_images(batch3)

# Run inference (this will process all images in the list)
preds3 = model.infer(
    views3,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()
last_pose3 = None

for i, pred in enumerate(preds3):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose2@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose3 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)


# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [18]:
o3d.visualization.draw_geometries(geometries)

In [20]:
batch1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",

]


views1 = load_images(batch1)

# Run inference (this will process all images in the list)
preds1 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

geometries = []
last_pose = None
pcd_batch1_overlap = None # We need to save this cloud

print("Processing Batch 1...")
for i, pred1 in enumerate(preds1):
    
    # Use 'pts3d' - these are already in batch 1's world frame
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_world)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    # Add camera pose for visualization
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    geometries.append(pcd1)
    geometries.append(camera_frame)

    if i == len(preds1) - 1:
        # This is frame_000200.jpg in World 1
        pcd_batch1_overlap = pcd1 
        last_pose = camera_pose # Save for visualization if needed

# --- BATCH 2 ---
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",

]

views2 = load_images(batch2)
preds2 = model.infer(views2)
# ...

pcd_batch2_overlap = None # This will be frame_000200.jpg in World 2
point_clouds_batch2 = []  # Store batch 2 clouds temporarily
camera_frames_batch2 = [] # Store batch 2 frames temporarily

print("Processing Batch 2 (in its own coordinate system)...")
for i, pred in enumerate(preds2):
    
    # Use 'pts3d' - these are in batch 2's world frame
    points_world_b2 = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors_b2 = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_world_b2)
    pcd.colors = o3d.utility.Vector3dVector(colors_b2)
    
    camera_pose_b2 = pred["camera_poses"].squeeze().cpu().numpy()
    camera_frame_b2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame_b2.transform(camera_pose_b2)

    if i == 0:
        # This is frame_000200.jpg in World 2
        pcd_batch2_overlap = pcd 
    else:
        # These are the *new* clouds (250, 300, 350, 400)
        point_clouds_batch2.append(pcd)
        camera_frames_batch2.append(camera_frame_b2)

# --- ALIGNMENT STEP (using ICP) ---

print("Aligning Batch 2 to Batch 1 using ICP...")
# Voxel downsample for faster ICP
voxel_size = 0.05 # Adjust this based on your scene's scale
source = pcd_batch2_overlap.voxel_down_sample(voxel_size)
target = pcd_batch1_overlap.voxel_down_sample(voxel_size)

# Set an initial guess (identity matrix)
trans_init = np.identity(4)

# Run ICP
# You may need to tune 'max_correspondence_distance'
reg_p2p = o3d.pipelines.registration.registration_icp(
    source, target, 0.2, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))

# Get the transformation matrix T that maps World 2 -> World 1
T_batch2_to_batch1 = reg_p2p.transformation
print("ICP transformation found:")
print(T_batch2_to_batch1)


# --- ADD BATCH 2 GEOMETRIES (NOW TRANSFORMED) ---

print("Applying transformation to Batch 2...")
for pcd in point_clouds_batch2:
    pcd.transform(T_batch2_to_batch1)
    pcd.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(pcd)

for frame in camera_frames_batch2:
    frame.transform(T_batch2_to_batch1)
    frame.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(frame)

# Apply the Y/Z flip to batch 1 geometries
for i in range(len(predictions1) * 2): # 2 geometries (pcd, frame) per image
    geometries[i].transform(transform_matrix)

# --- VISUALIZE ---
print("Visualizing combined map...")
o3d.visualization.draw_geometries(geometries)

# # --- SAVE ---
# print(f"Saving combined PLY to {output_filename}...")
# combined_pcd = o3d.geometry.PointCloud()
# for geo in geometries:
#     if isinstance(geo, o3d.geometry.PointCloud):
#         combined_pcd += geo

# # Voxel downsample the final cloud for a reasonable file size
# final_pcd = combined_pcd.voxel_down_sample(voxel_size=0.02)
# o3d.io.write_point_cloud(output_filename, final_pcd)
# print("Done.")

Processing Batch 1...
Processing Batch 2 (in its own coordinate system)...
Aligning Batch 2 to Batch 1 using ICP...
ICP transformation found:
[[ 0.99999367  0.00335097 -0.00119958  0.07746477]
 [-0.00335421  0.9999907  -0.00271146  0.05520372]
 [ 0.00119049  0.00271546  0.9999956   0.06450189]
 [ 0.          0.          0.          1.        ]]
Applying transformation to Batch 2...
Visualizing combined map...


In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Imports ---
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np
import copy

# =====================
# Params (tune as needed)
# =====================
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_global_filtered.ply"

# Geometry / ICP tuning
VOXEL_SIZE = 0.07                     # a bit coarser helps SNR in outdoor scenes
ICP_DISTANCE_THRESH = VOXEL_SIZE * 2  # fine-level corr radius
ICP_ESTIMATION = o3d.pipelines.registration.TransformationEstimationPointToPlane()
ICP_CRITERIA = o3d.pipelines.registration.ICPConvergenceCriteria(
    relative_fitness=1e-6, relative_rmse=1e-6, max_iteration=200
)

# Edge acceptance thresholds
MIN_ICP_FITNESS = 0.10   # relax slightly; loop closure will fix residual drift
MAX_ICP_RMSE    = 0.15

# Downweighting factor for marginal edges (applied to Info matrix)
MARGINAL_EDGE_FITNESS = 0.20
MARGINAL_INFO_SCALE   = 0.3

# Distance filter per-frame (drop far points)
MAX_FILTER_DISTANCE = 20.0  # meters

# Final coordinate flip (apply ONCE, at the very end)
AXIS_FLIP = np.array([
    [1,  0,  0,  0],   # X
    [0, -1,  0,  0],   # Y
    [0,  0, -1,  0],   # Z
    [0,  0,  0,  1]
])

# Overlap submap sizes (number of frames used on each side)
SUBMAP_K = 2  # last 2 from previous batch vs first 2 from next batch

# =====================
# Data Batches (your paths)
# =====================
images1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
]
images2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
]
images3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
]

# =====================
# Helpers
# =====================
def process_batch(image_paths, model, max_distance):
    """
    Inference on a batch -> list of filtered Open3D PointClouds (world frame) + camera poses.
    Returns:
        pcds:  [o3d.geometry.PointCloud, ...]  (len == len(image_paths))
        poses: [4x4 np.ndarray, ...]           pose of each frame in that batch's local world
    """
    print(f"  Inferring batch of {len(image_paths)} images...")
    views = load_images(image_paths)
    preds = model.infer(views, use_amp=True, amp_dtype="bf16")

    pcds, poses = [], []
    for pred in preds:
        pts = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        rgb = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        T_w = pred["camera_poses"].squeeze().cpu().numpy()  # 4x4

        # distance filter from camera origin (in that frame's world)
        cam_o = T_w[:3, 3]
        d = np.linalg.norm(pts - cam_o, axis=1)
        mask = d <= max_distance
        pts_f = pts[mask]
        rgb_f = rgb[mask]

        p = o3d.geometry.PointCloud()
        p.points = o3d.utility.Vector3dVector(pts_f)
        p.colors = o3d.utility.Vector3dVector(rgb_f)

        pcds.append(p)
        poses.append(T_w)

    return pcds, poses

def preprocess(pcd, voxel_size):
    q = pcd.voxel_down_sample(voxel_size)
    if q.has_points():
        q.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
        )
    return q

def icp_once(src, tgt, max_corr, init):
    return o3d.pipelines.registration.registration_icp(
        src, tgt, max_corr, init, ICP_ESTIMATION, ICP_CRITERIA
    )

def run_icp_multiscale_with_info(source, target, init_transform):
    """
    Multiscale ICP (coarse->fine) with growing correspondence radii at coarse levels.
    Returns (ok, T, Info, fitness, rmse).
    """
    scales = [VOXEL_SIZE*3, VOXEL_SIZE*2, VOXEL_SIZE]
    T = init_transform.copy()
    reg = None

    for vs in scales:
        src = preprocess(source, vs)
        tgt = preprocess(target, vs)
        if not src.has_points() or not tgt.has_points():
            print("  [Warn] ICP aborted: empty downsampled cloud(s).")
            return False, np.eye(4), np.zeros((6,6)), 0.0, np.inf

        max_corr = max(ICP_DISTANCE_THRESH, 2.5 * vs)
        reg = icp_once(src, tgt, max_corr, T)
        T = reg.transformation

    fitness = reg.fitness
    rmse = reg.inlier_rmse
    ok = (fitness >= MIN_ICP_FITNESS) and (rmse <= MAX_ICP_RMSE)

    # Info computed at fine scale
    src_f = preprocess(source, VOXEL_SIZE)
    tgt_f = preprocess(target, VOXEL_SIZE)
    Info = o3d.pipelines.registration.get_information_matrix_from_point_clouds(
        src_f, tgt_f, ICP_DISTANCE_THRESH, T
    )
    if not ok:
        print(f"  [Warn] ICP low quality (fitness={fitness:.3f}, rmse={rmse:.3f}).")
    return ok, T, Info, fitness, rmse

def make_submap(pcd_list, indices, voxel_size):
    """
    Merge multiple frames into a small submap.
    """
    sub = o3d.geometry.PointCloud()
    for idx in indices:
        if 0 <= idx < len(pcd_list) and pcd_list[idx].has_points():
            sub += pcd_list[idx]
    if not sub.has_points():
        return sub
    sub = sub.voxel_down_sample(voxel_size * 0.5)  # a tad denser for features
    sub.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    return sub

def add_edge_with_downweight(pose_graph, i, j, T_ji, Info, fitness):
    """
    Adds PoseGraphEdge(i->j) with optional downweighting based on fitness.
    """
    if fitness < MARGINAL_EDGE_FITNESS:
        Info = Info * MARGINAL_INFO_SCALE
        print(f"    [Info] Downweighting edge {i}->{j} (fitness={fitness:.3f}).")
    pose_graph.edges.append(
        o3d.pipelines.registration.PoseGraphEdge(i, j, T_ji, Info, uncertain=(abs(i-j) > 1))
    )

# =====================
# Main
# =====================
def main():
    # Device & model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading MapAnything model...")
    model = MapAnything.from_pretrained("facebook/map-anything").to(device)

    # 1) Inference
    print("\n--- Step 1: Processing all batches ---")
    print(f"Applying distance filter: <= {MAX_FILTER_DISTANCE} m")
    all_pcds_b1, poses_b1 = process_batch(images1, model, MAX_FILTER_DISTANCE)
    all_pcds_b2, poses_b2 = process_batch(images2, model, MAX_FILTER_DISTANCE)
    all_pcds_b3, poses_b3 = process_batch(images3, model, MAX_FILTER_DISTANCE)
    all_batches = [all_pcds_b1, all_pcds_b2, all_pcds_b3]
    all_poses   = [poses_b1, poses_b2, poses_b3]

    # 2) Pose graph init (Open3D nodes store **inverse** of world pose)
    print("\n--- Step 2: Build initial poses & nodes ---")
    T_w_0 = np.eye(4)
    poses_w = [T_w_0]  # forward world poses for nodes 0,1,2

    pose_graph = o3d.pipelines.registration.PoseGraph()
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_0)))

    # 3) Sequential constraints (0->1 at f=200; 1->2 at f=400), with pose-diff init + submaps
    print("\n--- Step 3: Sequential constraints (0->1, 1->2) ---")

    # -------- 0 -> 1 (use last-of-B1 vs first-of-B2; f=200) --------
    print("  ICP for Batch1<->Batch2 (overlap @ 200) with pose-diff init + submaps...")
    # Submaps
    sub1 = make_submap(all_pcds_b1, indices=list(range(len(all_pcds_b1)-SUBMAP_K, len(all_pcds_b1))), voxel_size=VOXEL_SIZE)
    sub2 = make_submap(all_pcds_b2, indices=list(range(0, min(SUBMAP_K, len(all_pcds_b2)))), voxel_size=VOXEL_SIZE)
    if not sub1.has_points() or not sub2.has_points():
        print("    [Error] Empty submaps for 0->1; abort.")
        return

    # Pose-diff init from the shared frame index (here: last of b1 vs first of b2)
    T_w1_f200 = poses_b1[-1]
    T_w2_f200 = poses_b2[0]
    T_2_to_1_init = T_w1_f200 @ np.linalg.inv(T_w2_f200)

    ok21, T_2_to_1, Info_2_to_1, fit21, rmse21 = run_icp_multiscale_with_info(
        source=sub2, target=sub1, init_transform=T_2_to_1_init
    )
    print(f"    fitness={fit21:.3f}, rmse={rmse21:.3f}")
    if not ok21:
        print("    [Error] Sequential edge 0->1 is too weak; aborting to avoid broken graph.")
        return

    # Global pose for node 1 in world-0
    T_w_1 = poses_w[0] @ T_2_to_1
    poses_w.append(T_w_1)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_1)))
    add_edge_with_downweight(pose_graph, 0, 1, T_2_to_1, Info_2_to_1, fit21)

    # -------- 1 -> 2 (use last-of-B2 vs first-of-B3; f=400) --------
    print("  ICP for Batch2<->Batch3 (overlap @ 400) with pose-diff init + submaps...")
    sub2_b = make_submap(all_pcds_b2, indices=list(range(len(all_pcds_b2)-SUBMAP_K, len(all_pcds_b2))), voxel_size=VOXEL_SIZE)
    sub3_b = make_submap(all_pcds_b3, indices=list(range(0, min(SUBMAP_K, len(all_pcds_b3)))), voxel_size=VOXEL_SIZE)
    if not sub2_b.has_points() or not sub3_b.has_points():
        print("    [Error] Empty submaps for 1->2; abort.")
        return

    T_w2_f400 = poses_b2[-1]
    T_w3_f400 = poses_b3[0]
    T_3_to_2_init = T_w2_f400 @ np.linalg.inv(T_w3_f400)

    ok32, T_3_to_2, Info_3_to_2, fit32, rmse32 = run_icp_multiscale_with_info(
        source=sub3_b, target=sub2_b, init_transform=T_3_to_2_init
    )
    print(f"    fitness={fit32:.3f}, rmse={rmse32:.3f}")
    if not ok32:
        print("    [Error] Sequential edge 1->2 is too weak; aborting to avoid broken graph.")
        return

    T_w_2 = T_w_1 @ T_3_to_2
    poses_w.append(T_w_2)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_2)))
    add_edge_with_downweight(pose_graph, 1, 2, T_3_to_2, Info_3_to_2, fit32)

    # 4) Loop closure (2 -> 0), end-of-B3 vs start-of-B1, with good init
    print("\n--- Step 4: Loop closure (2->0) ---")
    # Make slightly larger submaps for loop if you like
    sub3_end = make_submap(all_pcds_b3, indices=list(range(max(0, len(all_pcds_b3)-SUBMAP_K-1), len(all_pcds_b3))), voxel_size=VOXEL_SIZE)
    sub1_start = make_submap(all_pcds_b1, indices=list(range(0, min(SUBMAP_K+1, len(all_pcds_b1)))), voxel_size=VOXEL_SIZE)

    # Good init: current odometry pose from chain (T_w_2), expressed as transform to node 0
    # We want T_3->1; approximate init can be T_w_0 @ inv(T_w_2) but better is using model poses if you have a known same-scene pair.
    # We'll use odom pose as init (brings clouds roughly into place).
    T_loop_init = np.linalg.inv(T_w_2)  # transform from node2 world approx to node0
    okL, T_3_to_1, Info_loop, fitL, rmseL = run_icp_multiscale_with_info(
        source=sub3_end, target=sub1_start, init_transform=T_loop_init
    )
    print(f"    fitness={fitL:.3f}, rmse={rmseL:.3f}")
    if okL:
        add_edge_with_downweight(pose_graph, 2, 0, T_3_to_1, Info_loop, fitL)
    else:
        print("    [Warn] Loop edge skipped (low quality). Graph will still optimize.")

    # 5) Global optimization
    print("\n--- Step 5: Pose-graph optimization ---")
    option = o3d.pipelines.registration.GlobalOptimizationOption(
        max_correspondence_distance=ICP_DISTANCE_THRESH,
        edge_prune_threshold=0.25,
        reference_node=0
    )
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option
    )

    # 6) Fuse all clouds with optimized poses (invert node pose back to forward)
    print("\n--- Step 6: Combine map with optimized poses ---")
    pcd_combined = o3d.geometry.PointCloud()
    for i, batch in enumerate(all_batches):
        T_w_i_opt = np.linalg.inv(pose_graph.nodes[i].pose)
        # skip first (overlap) frame of later batches to avoid duplicates
        start_idx = 1 if i > 0 else 0
        for p in batch[start_idx:]:
            if not p.has_points():
                continue
            q = copy.deepcopy(p)
            q.transform(T_w_i_opt)
            pcd_combined += q

    # final single axis-flip applied ONCE
    pcd_combined.transform(AXIS_FLIP)

    # 7) Visualize & (optionally) save
    print("\n--- Step 7: Visualization & Save ---")
    final_pcd = pcd_combined.voxel_down_sample(voxel_size=VOXEL_SIZE)
    print(f"Final combined point cloud has {len(final_pcd.points)} points.")
    o3d.visualization.draw_geometries([final_pcd])

    # Uncomment to write to disk
    # print(f"Saving combined PLY to {output_filename} ...")
    # o3d.io.write_point_cloud(output_filename, final_pcd)
    # print("Done.")

if __name__ == "__main__":
    main()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading MapAnything model...
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main



--- Step 1: Processing all batches ---
Applying distance filter: <= 20.0 m
  Inferring batch of 5 images...
  Inferring batch of 5 images...
  Inferring batch of 5 images...

--- Step 2: Build initial poses & nodes ---

--- Step 3: Sequential constraints (0->1, 1->2) ---
  ICP for Batch1<->Batch2 (overlap @ 200) with pose-diff init + submaps...
    fitness=0.765, rmse=0.078
  ICP for Batch2<->Batch3 (overlap @ 400) with pose-diff init + submaps...
    fitness=0.448, rmse=0.078

--- Step 4: Loop closure (2->0) ---
  [Warn] ICP low quality (fitness=0.000, rmse=0.000).
    fitness=0.000, rmse=0.000
    [Warn] Loop edge skipped (low quality). Graph will still optimize.

--- Step 5: Pose-graph optimization ---

--- Step 6: Combine map with optimized poses ---

--- Step 7: Visualization & Save ---
Final combined point cloud has 167399 points.


In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Imports ---
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np
import copy

# =====================
# Params (tune as needed)
# =====================
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_global_filtered.ply"

# Geometry / ICP tuning
VOXEL_SIZE = 0.07                     # 0.07–0.10 is robust outdoors
ICP_DISTANCE_THRESH = VOXEL_SIZE * 2  # fine-level corr radius
ICP_ESTIMATION = o3d.pipelines.registration.TransformationEstimationPointToPlane()
ICP_CRITERIA = o3d.pipelines.registration.ICPConvergenceCriteria(
    relative_fitness=1e-6, relative_rmse=1e-6, max_iteration=200
)

# Edge acceptance thresholds
MIN_ICP_FITNESS = 0.10   # relax slightly; loop closure will fix residual drift
MAX_ICP_RMSE    = 0.15

# Downweighting for marginal edges
MARGINAL_EDGE_FITNESS = 0.20
MARGINAL_INFO_SCALE   = 0.3

# Distance filter per-frame (drop far points)
MAX_FILTER_DISTANCE = 20.0  # meters; try 15.0 if far noise hurts ICP

# Optional height crop (disabled by default)
USE_Z_CROP = False
Z_MIN, Z_MAX = -5.0, 5.0    # set if you enable USE_Z_CROP

# Final coordinate flip (apply ONCE, at the very end)
AXIS_FLIP = np.array([
    [1,  0,  0,  0],   # X
    [0, -1,  0,  0],   # Y
    [0,  0, -1,  0],   # Z
    [0,  0,  0,  1]
])

# Overlap submap sizes (number of frames used on each side)
SUBMAP_K = 2  # last 2 of previous batch vs first 2 of next batch

# =====================
# Data Batches (your paths)
# =====================
images1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
]
images2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
]
images3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
]

images4 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000650.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000700.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000750.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000800.jpg",
]

# =====================
# Helpers
# =====================
def crop_z(pcd, zmin, zmax):
    if not pcd.has_points():
        return pcd
    pts = np.asarray(pcd.points)
    mask = (pts[:, 2] > zmin) & (pts[:, 2] < zmax)
    if mask.sum() == 0:
        return o3d.geometry.PointCloud()
    q = o3d.geometry.PointCloud()
    q.points = o3d.utility.Vector3dVector(pts[mask])
    if pcd.has_colors():
        q.colors = o3d.utility.Vector3dVector(np.asarray(pcd.colors)[mask])
    return q

def process_batch(image_paths, model, max_distance):
    """
    Inference on a batch -> list of filtered Open3D PointClouds (world frame) + camera poses.
    Returns:
        pcds:  [o3d.geometry.PointCloud, ...]
        poses:[4x4 np.ndarray, ...] pose of each frame in that batch's local world
    """
    print(f"  Inferring batch of {len(image_paths)} images...")
    views = load_images(image_paths)
    preds = model.infer(views, use_amp=True, amp_dtype="bf16")

    pcds, poses = [], []
    for pred in preds:
        pts = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        rgb = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        T_w = pred["camera_poses"].squeeze().cpu().numpy()  # 4x4

        # distance filter from camera origin (in that frame's world)
        cam_o = T_w[:3, 3]
        d = np.linalg.norm(pts - cam_o, axis=1)
        mask = d <= max_distance
        pts_f = pts[mask]
        rgb_f = rgb[mask]

        p = o3d.geometry.PointCloud()
        p.points = o3d.utility.Vector3dVector(pts_f)
        p.colors = o3d.utility.Vector3dVector(rgb_f)

        if USE_Z_CROP:
            p = crop_z(p, Z_MIN, Z_MAX)

        pcds.append(p)
        poses.append(T_w)

    return pcds, poses

def preprocess(pcd, voxel_size):
    q = pcd.voxel_down_sample(voxel_size)
    if q.has_points():
        q.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
        )
    return q

def icp_once(src, tgt, max_corr, init):
    return o3d.pipelines.registration.registration_icp(
        src, tgt, max_corr, init, ICP_ESTIMATION, ICP_CRITERIA
    )

def run_icp_multiscale_with_info(source, target, init_transform):
    """
    Multiscale ICP (coarse->fine) with growing corr radii at coarse levels.
    Returns (ok, T, Info, fitness, rmse).
    """
    scales = [VOXEL_SIZE*3, VOXEL_SIZE*2, VOXEL_SIZE]
    T = init_transform.copy()
    reg = None

    for vs in scales:
        src = preprocess(source, vs)
        tgt = preprocess(target, vs)
        if not src.has_points() or not tgt.has_points():
            print("  [Warn] ICP aborted: empty downsampled cloud(s).")
            return False, np.eye(4), np.zeros((6,6)), 0.0, np.inf

        max_corr = max(ICP_DISTANCE_THRESH, 2.5 * vs)
        reg = icp_once(src, tgt, max_corr, T)
        T = reg.transformation

    fitness = reg.fitness
    rmse = reg.inlier_rmse
    ok = (fitness >= MIN_ICP_FITNESS) and (rmse <= MAX_ICP_RMSE)

    # Info computed at fine scale
    src_f = preprocess(source, VOXEL_SIZE)
    tgt_f = preprocess(target, VOXEL_SIZE)
    Info = o3d.pipelines.registration.get_information_matrix_from_point_clouds(
        src_f, tgt_f, ICP_DISTANCE_THRESH, T
    )
    if not ok:
        print(f"  [Warn] ICP low quality (fitness={fitness:.3f}, rmse={rmse:.3f}).")
    return ok, T, Info, fitness, rmse

def make_submap(pcd_list, indices, voxel_size):
    """
    Merge multiple frames into a small submap.
    """
    sub = o3d.geometry.PointCloud()
    for idx in indices:
        if 0 <= idx < len(pcd_list) and pcd_list[idx].has_points():
            sub += pcd_list[idx]
    if not sub.has_points():
        return sub
    sub = sub.voxel_down_sample(voxel_size * 0.5)  # a tad denser for features
    sub.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    return sub

def add_edge_with_downweight(pose_graph, i, j, T_ji, Info, fitness):
    """
    Adds PoseGraphEdge(i->j) with optional downweighting based on fitness.
    """
    if fitness < MARGINAL_EDGE_FITNESS:
        Info = Info * MARGINAL_INFO_SCALE
        print(f"    [Info] Downweighting edge {i}->{j} (fitness={fitness:.3f}).")
    pose_graph.edges.append(
        o3d.pipelines.registration.PoseGraphEdge(i, j, T_ji, Info, uncertain=(abs(i-j) > 1))
    )

def coarse_global_reg(src, tgt, vs):
    """
    Optional coarse global registration (FPFH RANSAC) to help loop closure init.
    """
    src_d = preprocess(src, vs)
    tgt_d = preprocess(tgt, vs)
    if not src_d.has_points() or not tgt_d.has_points():
        return None, 0.0
    src_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        src_d, o3d.geometry.KDTreeSearchParamHybrid(radius=vs*5, max_nn=100))
    tgt_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        tgt_d, o3d.geometry.KDTreeSearchParamHybrid(radius=vs*5, max_nn=100))
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        src_d, tgt_d, src_fpfh, tgt_fpfh, mutual_filter=True,
        max_correspondence_distance=vs*3.5,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(vs*3.5)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(400000, 500)
    )
    if result is None:
        return None, 0.0
    return result.transformation, result.fitness

# =====================
# Main
# =====================
def main():
    # Device & model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading MapAnything model...")
    model = MapAnything.from_pretrained("facebook/map-anything").to(device)

    # 1) Inference
    print("\n--- Step 1: Processing all batches ---")
    print(f"Applying distance filter: <= {MAX_FILTER_DISTANCE} m")
    all_pcds_b1, poses_b1 = process_batch(images1, model, MAX_FILTER_DISTANCE)
    all_pcds_b2, poses_b2 = process_batch(images2, model, MAX_FILTER_DISTANCE)
    all_pcds_b3, poses_b3 = process_batch(images3, model, MAX_FILTER_DISTANCE)
    all_batches = [all_pcds_b1, all_pcds_b2, all_pcds_b3]

    # 2) Pose graph init (Open3D nodes store **inverse** of world pose)
    print("\n--- Step 2: Build initial poses & nodes ---")
    T_w_0 = np.eye(4)
    poses_w = [T_w_0]  # forward world poses for nodes 0,1,2

    pose_graph = o3d.pipelines.registration.PoseGraph()
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_0)))

    # 3) Sequential constraints (0->1 at f=200; 1->2 at f=400), with pose-diff init + submaps
    print("\n--- Step 3: Sequential constraints (0->1, 1->2) ---")

    # -------- 0 -> 1 (use last-of-B1 vs first-of-B2; f=200) --------
    print("  ICP for Batch1<->Batch2 (overlap @ 200) with pose-diff init + submaps...")
    sub1 = make_submap(all_pcds_b1, indices=list(range(len(all_pcds_b1)-SUBMAP_K, len(all_pcds_b1))), voxel_size=VOXEL_SIZE)
    sub2 = make_submap(all_pcds_b2, indices=list(range(0, min(SUBMAP_K, len(all_pcds_b2)))), voxel_size=VOXEL_SIZE)
    if not sub1.has_points() or not sub2.has_points():
        print("    [Error] Empty submaps for 0->1; abort.")
        return

    # Pose-diff init from the shared frame index (last of b1 vs first of b2)
    T_w1_f200 = poses_b1[-1]
    T_w2_f200 = poses_b2[0]
    T_2_to_1_init = T_w1_f200 @ np.linalg.inv(T_w2_f200)

    ok21, T_2_to_1, Info_2_to_1, fit21, rmse21 = run_icp_multiscale_with_info(
        source=sub2, target=sub1, init_transform=T_2_to_1_init
    )
    print(f"    fitness={fit21:.3f}, rmse={rmse21:.3f}")
    if not ok21:
        print("    [Error] Sequential edge 0->1 is too weak; aborting to avoid broken graph.")
        return

    # Global pose for node 1 in world-0
    T_w_1 = poses_w[0] @ T_2_to_1
    poses_w.append(T_w_1)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_1)))
    add_edge_with_downweight(pose_graph, 0, 1, T_2_to_1, Info_2_to_1, fit21)

    # Extra constraint near 200 (B1[-2] ↔ B2[1])
    sub1_alt = make_submap(all_pcds_b1, indices=[max(0, len(all_pcds_b1)-2)], voxel_size=VOXEL_SIZE)
    sub2_alt = make_submap(all_pcds_b2, indices=[1], voxel_size=VOXEL_SIZE)
    if sub1_alt.has_points() and sub2_alt.has_points():
        T_init = poses_b1[-2] @ np.linalg.inv(poses_b2[1])
        ok, T, Info, fit, rmse = run_icp_multiscale_with_info(source=sub2_alt, target=sub1_alt, init_transform=T_init)
        print(f"    extra 0->1 edge fitness={fit:.3f}, rmse={rmse:.3f}")
        if ok:
            add_edge_with_downweight(pose_graph, 0, 1, T, Info, fit)

    # -------- 1 -> 2 (use last-of-B2 vs first-of-B3; f=400) --------
    print("  ICP for Batch2<->Batch3 (overlap @ 400) with pose-diff init + submaps...")
    sub2_b = make_submap(all_pcds_b2, indices=list(range(len(all_pcds_b2)-SUBMAP_K, len(all_pcds_b2))), voxel_size=VOXEL_SIZE)
    sub3_b = make_submap(all_pcds_b3, indices=list(range(0, min(SUBMAP_K, len(all_pcds_b3)))), voxel_size=VOXEL_SIZE)
    if not sub2_b.has_points() or not sub3_b.has_points():
        print("    [Error] Empty submaps for 1->2; abort.")
        return

    T_w2_f400 = poses_b2[-1]
    T_w3_f400 = poses_b3[0]
    T_3_to_2_init = T_w2_f400 @ np.linalg.inv(T_w3_f400)

    ok32, T_3_to_2, Info_3_to_2, fit32, rmse32 = run_icp_multiscale_with_info(
        source=sub3_b, target=sub2_b, init_transform=T_3_to_2_init
    )
    print(f"    fitness={fit32:.3f}, rmse={rmse32:.3f}")
    if not ok32:
        print("    [Error] Sequential edge 1->2 is too weak; aborting to avoid broken graph.")
        return

    T_w_2 = T_w_1 @ T_3_to_2
    poses_w.append(T_w_2)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_2)))
    add_edge_with_downweight(pose_graph, 1, 2, T_3_to_2, Info_3_to_2, fit32)

    # Extra constraint near 400 (B2[-2] ↔ B3[1])
    sub2_alt = make_submap(all_pcds_b2, indices=[max(0, len(all_pcds_b2)-2)], voxel_size=VOXEL_SIZE)
    sub3_alt = make_submap(all_pcds_b3, indices=[1], voxel_size=VOXEL_SIZE)
    if sub2_alt.has_points() and sub3_alt.has_points():
        T_init = poses_b2[-2] @ np.linalg.inv(poses_b3[1])
        ok, T, Info, fit, rmse = run_icp_multiscale_with_info(source=sub3_alt, target=sub2_alt, init_transform=T_init)
        print(f"    extra 1->2 edge fitness={fit:.3f}, rmse={rmse:.3f}")
        if ok:
            add_edge_with_downweight(pose_graph, 1, 2, T, Info, fit)

    # 4) Loop closure (2 -> 0): pick truly overlapping frames; use pose-diff init + optional coarse reg
    print("\n--- Step 4: Loop closure (2->0) ---")
    i1, i3 = 0, -1  # try B1 first, B3 last; adjust if your route didn't return to start

    sub1_start = make_submap(all_pcds_b1, indices=[i1, min(i1+1, len(all_pcds_b1)-1)], voxel_size=VOXEL_SIZE)
    sub3_end   = make_submap(all_pcds_b3, indices=[max(0, len(all_pcds_b3)-2), len(all_pcds_b3)-1], voxel_size=VOXEL_SIZE)

    # pose-diff initialization from the model's camera poses at those frames
    T_w1 = poses_b1[i1]
    T_w3 = poses_b3[i3]
    T_3_to_1_init = T_w1 @ np.linalg.inv(T_w3)

    # Optional coarse global (FPFH RANSAC) to help if init is off
    T_coarse, coarse_fit = coarse_global_reg(sub3_end, sub1_start, VOXEL_SIZE*2.0)
    if T_coarse is not None and coarse_fit > 0.05:
        print("    [Info] Using coarse global registration as init for loop.")
        T_init = T_coarse
    else:
        T_init = T_3_to_1_init

    okL, T_3_to_1, Info_loop, fitL, rmseL = run_icp_multiscale_with_info(
        source=sub3_end, target=sub1_start, init_transform=T_init
    )
    print(f"    loop fitness={fitL:.3f}, rmse={rmseL:.3f}")
    loop_added = False
    if okL:
        add_edge_with_downweight(pose_graph, 2, 0, T_3_to_1, Info_loop, fitL)
        loop_added = True
    else:
        print("    [Warn] Loop edge still weak; consider trying different frame pairs.")

    # Soft absolute prior (very weak) if no true loop landed
    if not loop_added:
        T_soft = poses_b1[0] @ np.linalg.inv(poses_b3[-1])  # model-based guess
        Info_soft = np.eye(6) * 0.1                         # weak prior
        print("    [Info] Adding soft 2->0 prior (no true loop).")
        pose_graph.edges.append(
            o3d.pipelines.registration.PoseGraphEdge(2, 0, T_soft, Info_soft, uncertain=True)
        )

    # 5) Global optimization
    print("\n--- Step 5: Pose-graph optimization ---")
    option = o3d.pipelines.registration.GlobalOptimizationOption(
        max_correspondence_distance=ICP_DISTANCE_THRESH,
        edge_prune_threshold=0.25,
        reference_node=0
    )
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option
    )

    # 6) Fuse all clouds with optimized poses (invert node pose back to forward)
    print("\n--- Step 6: Combine map with optimized poses ---")
    pcd_combined = o3d.geometry.PointCloud()
    for i, batch in enumerate(all_batches):
        T_w_i_opt = np.linalg.inv(pose_graph.nodes[i].pose)
        # skip first (overlap) frame of later batches to avoid duplicates
        start_idx = 1 if i > 0 else 0
        for p in batch[start_idx:]:
            if not p.has_points():
                continue
            q = copy.deepcopy(p)
            q.transform(T_w_i_opt)
            pcd_combined += q

    # final single axis-flip applied ONCE
    pcd_combined.transform(AXIS_FLIP)

    # 7) Visualize & (optionally) save
    print("\n--- Step 7: Visualization & Save ---")
    final_pcd = pcd_combined.voxel_down_sample(voxel_size=VOXEL_SIZE)
    print(f"Final combined point cloud has {len(final_pcd.points)} points.")
    o3d.visualization.draw_geometries([final_pcd])

    # Uncomment to write to disk
    # print(f"Saving combined PLY to {output_filename} ...")
    # o3d.io.write_point_cloud(output_filename, final_pcd)
    # print("Done.")

if __name__ == "__main__":
    main()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading MapAnything model...
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main



--- Step 1: Processing all batches ---
Applying distance filter: <= 20.0 m
  Inferring batch of 5 images...
  Inferring batch of 5 images...
  Inferring batch of 5 images...

--- Step 2: Build initial poses & nodes ---

--- Step 3: Sequential constraints (0->1, 1->2) ---
  ICP for Batch1<->Batch2 (overlap @ 200) with pose-diff init + submaps...
    fitness=0.765, rmse=0.078
    extra 0->1 edge fitness=0.677, rmse=0.077
  ICP for Batch2<->Batch3 (overlap @ 400) with pose-diff init + submaps...
    fitness=0.448, rmse=0.078
    extra 1->2 edge fitness=0.441, rmse=0.078

--- Step 4: Loop closure (2->0) ---
    [Info] Using coarse global registration as init for loop.
    loop fitness=0.461, rmse=0.084

--- Step 5: Pose-graph optimization ---

--- Step 6: Combine map with optimized poses ---

--- Step 7: Visualization & Save ---
Final combined point cloud has 167694 points.


In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Imports ---
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np
import copy

# =====================
# Params (tune as needed)
# =====================
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_global_filtered.ply"

# Pose / axis convention guards
POSE_CONVENTION = "c2w"   # set to "w2c" if model returns world->camera
APPLY_AXIS_FLIP = False   # keep False until map looks correct; then enable if needed
AXIS_FLIP = np.array([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]])

# Geometry / ICP tuning
VOXEL_SIZE = 0.07
ICP_DISTANCE_THRESH = VOXEL_SIZE * 2
ICP_ESTIMATION = o3d.pipelines.registration.TransformationEstimationPointToPlane()
ICP_CRITERIA = o3d.pipelines.registration.ICPConvergenceCriteria(
    relative_fitness=1e-6, relative_rmse=1e-6, max_iteration=200
)

# Edge acceptance thresholds
MIN_ICP_FITNESS = 0.10
MAX_ICP_RMSE    = 0.15

# Downweighting for marginal edges
MARGINAL_EDGE_FITNESS = 0.20
MARGINAL_INFO_SCALE   = 0.3

# Distance filter per-frame (drop far points)
MAX_FILTER_DISTANCE = 30.0

# Optional height crop (disabled by default)
USE_Z_CROP = False
Z_MIN, Z_MAX = -5.0, 5.0

# Overlap submap sizes (number of frames used on each side)
SUBMAP_K = 2  # last 2 prev batch vs first 2 next batch

# =====================
# Data Batches (your paths)
# =====================
images1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
]
images2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
]
images3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
]
images4 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000650.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000700.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000750.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000800.jpg",
]
images5 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000800.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000850.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000900.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000950.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_001000.jpg",
]


# =====================
# Helpers
# =====================
def normalize_pose(T):
    """Return camera->world pose no matter what the model outputs."""
    return T if POSE_CONVENTION.lower()=="c2w" else np.linalg.inv(T)

def crop_z(pcd, zmin, zmax):
    if not pcd.has_points():
        return pcd
    pts = np.asarray(pcd.points)
    mask = (pts[:, 2] > zmin) & (pts[:, 2] < zmax)
    if mask.sum() == 0:
        return o3d.geometry.PointCloud()
    q = o3d.geometry.PointCloud()
    q.points = o3d.utility.Vector3dVector(pts[mask])
    if pcd.has_colors():
        q.colors = o3d.utility.Vector3dVector(np.asarray(pcd.colors)[mask])
    return q

def process_batch(image_paths, model, max_distance):
    """
    Inference on a batch -> list of filtered Open3D PointClouds (world frame) + camera poses (normalized c2w).
    """
    print(f"  Inferring batch of {len(image_paths)} images...")
    views = load_images(image_paths)
    preds = model.infer(views, use_amp=True, amp_dtype="bf16")

    pcds, poses = [], []
    for pred in preds:
        pts = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        rgb = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        T_raw = pred["camera_poses"].squeeze().cpu().numpy()  # 4x4
        T_w = normalize_pose(T_raw)

        # distance filter from camera origin
        cam_o = T_w[:3, 3]
        d = np.linalg.norm(pts - cam_o, axis=1)
        mask = d <= max_distance
        pts_f = pts[mask]
        rgb_f = rgb[mask]

        p = o3d.geometry.PointCloud()
        p.points = o3d.utility.Vector3dVector(pts_f)
        p.colors = o3d.utility.Vector3dVector(rgb_f)
        if USE_Z_CROP:
            p = crop_z(p, Z_MIN, Z_MAX)

        pcds.append(p)
        poses.append(T_w)

    return pcds, poses

def preprocess(pcd, voxel_size):
    q = pcd.voxel_down_sample(voxel_size)
    if q.has_points():
        q.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
        )
    return q

def icp_once(src, tgt, max_corr, init):
    return o3d.pipelines.registration.registration_icp(
        src, tgt, max_corr, init, ICP_ESTIMATION, ICP_CRITERIA
    )

def run_icp_multiscale_with_info(source, target, init_transform):
    """
    Multiscale ICP (coarse->fine) with growing corr radii at coarse levels.
    Returns (ok, T, Info, fitness, rmse).
    """
    scales = [VOXEL_SIZE*3, VOXEL_SIZE*2, VOXEL_SIZE]
    T = init_transform.copy()
    reg = None

    for vs in scales:
        src = preprocess(source, vs)
        tgt = preprocess(target, vs)
        if not src.has_points() or not tgt.has_points():
            print("  [Warn] ICP aborted: empty downsampled cloud(s).")
            return False, np.eye(4), np.zeros((6,6)), 0.0, np.inf

        max_corr = max(ICP_DISTANCE_THRESH, 2.5 * vs)
        reg = icp_once(src, tgt, max_corr, T)
        T = reg.transformation

    fitness = reg.fitness
    rmse = reg.inlier_rmse
    ok = (fitness >= MIN_ICP_FITNESS) and (rmse <= MAX_ICP_RMSE)

    # Info computed at fine scale
    src_f = preprocess(source, VOXEL_SIZE)
    tgt_f = preprocess(target, VOXEL_SIZE)
    Info = o3d.pipelines.registration.get_information_matrix_from_point_clouds(
        src_f, tgt_f, ICP_DISTANCE_THRESH, T
    )
    if not ok:
        print(f"  [Warn] ICP low quality (fitness={fitness:.3f}, rmse={rmse:.3f}).")
    return ok, T, Info, fitness, rmse

def make_submap(pcd_list, indices, voxel_size):
    """
    Merge multiple frames into a small submap.
    """
    sub = o3d.geometry.PointCloud()
    for idx in indices:
        if 0 <= idx < len(pcd_list) and pcd_list[idx].has_points():
            sub += pcd_list[idx]
    if not sub.has_points():
        return sub
    sub = sub.voxel_down_sample(voxel_size * 0.5)  # a tad denser for features
    sub.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    return sub

def add_edge_with_downweight(pose_graph, i, j, T_ji, Info, fitness):
    """
    Adds PoseGraphEdge(i->j) with optional downweighting based on fitness.
    """
    if fitness < MARGINAL_EDGE_FITNESS:
        Info = Info * MARGINAL_INFO_SCALE
        print(f"    [Info] Downweighting edge {i}->{j} (fitness={fitness:.3f}).")
    pose_graph.edges.append(
        o3d.pipelines.registration.PoseGraphEdge(i, j, T_ji, Info, uncertain=(abs(i-j) > 1))
    )

def coarse_global_reg(src, tgt, vs):
    """
    Optional coarse global registration (FPFH RANSAC) to help loop closure init.
    """
    src_d = preprocess(src, vs)
    tgt_d = preprocess(tgt, vs)
    if not src_d.has_points() or not tgt_d.has_points():
        return None, 0.0
    src_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        src_d, o3d.geometry.KDTreeSearchParamHybrid(radius=vs*5, max_nn=100))
    tgt_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        tgt_d, o3d.geometry.KDTreeSearchParamHybrid(radius=vs*5, max_nn=100))
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        src_d, tgt_d, src_fpfh, tgt_fpfh, mutual_filter=True,
        max_correspondence_distance=vs*3.5,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(vs*3.5)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(400000, 500)
    )
    if result is None:
        return None, 0.0
    return result.transformation, result.fitness

# =====================
# Main
# =====================
def main():
    # Device & model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading MapAnything model...")
    model = MapAnything.from_pretrained("facebook/map-anything").to(device)

    # 1) Inference
    print("\n--- Step 1: Processing all batches ---")
    print(f"Applying distance filter: <= {MAX_FILTER_DISTANCE} m")
    all_pcds_b1, poses_b1 = process_batch(images1, model, MAX_FILTER_DISTANCE)
    all_pcds_b2, poses_b2 = process_batch(images2, model, MAX_FILTER_DISTANCE)
    all_pcds_b3, poses_b3 = process_batch(images3, model, MAX_FILTER_DISTANCE)
    all_pcds_b4, poses_b4 = process_batch(images4, model, MAX_FILTER_DISTANCE)
    all_pcds_b5, poses_b5 = process_batch(images5, model, MAX_FILTER_DISTANCE)


    # Pack to generic lists so code works for N batches
    all_batches = [all_pcds_b1, all_pcds_b2, all_pcds_b3, all_pcds_b4, all_pcds_b5]
    all_poses   = [poses_b1,    poses_b2,    poses_b3,    poses_b4,    poses_b5]
    num_nodes   = len(all_batches)


    # 2) Pose graph init (Open3D nodes store **inverse** of world pose)
    print("\n--- Step 2: Build initial poses & nodes ---")
    pose_graph = o3d.pipelines.registration.PoseGraph()
    T_w_nodes = [np.eye(4)]  # forward world poses
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_nodes[0])))

    # 3) Sequential constraints for all neighbors
    print("\n--- Step 3: Sequential constraints for neighbors ---")
    for i in range(num_nodes - 1):
        # Overlap: last of batch i vs first of batch i+1
        sub_i   = make_submap(all_batches[i],   list(range(len(all_batches[i])-SUBMAP_K, len(all_batches[i]))), VOXEL_SIZE)
        sub_ip1 = make_submap(all_batches[i+1], list(range(0, min(SUBMAP_K, len(all_batches[i+1])))), VOXEL_SIZE)
        if not sub_i.has_points() or not sub_ip1.has_points():
            print(f"  [Error] Empty submaps for {i}->{i+1}; abort.")
            return

        # Pose-diff init using shared real frame (last idx of i vs first idx of i+1)
        T_w_i_end  = all_poses[i][-1]
        T_w_ip1_0  = all_poses[i+1][0]
        T_ip1_to_i_init = T_w_i_end @ np.linalg.inv(T_w_ip1_0)

        ok, T_ip1_to_i, Info, fit, rmse = run_icp_multiscale_with_info(
            source=sub_ip1, target=sub_i, init_transform=T_ip1_to_i_init
        )
        print(f"  {i}->{i+1} fitness={fit:.3f}, rmse={rmse:.3f}")
        if not ok:
            print("    [Error] Sequential edge is weak; aborting to avoid a broken graph.")
            return

        # Accumulate global pose for node i+1
        T_w_next = T_w_nodes[i] @ T_ip1_to_i
        T_w_nodes.append(T_w_next)
        pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(np.linalg.inv(T_w_next)))
        add_edge_with_downweight(pose_graph, i, i+1, T_ip1_to_i, Info, fit)

        # Extra constraints around overlap (±1) if possible
        if len(all_batches[i]) >= 2 and len(all_batches[i+1]) >= 2:
            sub_i_alt   = make_submap(all_batches[i],   [max(0, len(all_batches[i]) - 2)], VOXEL_SIZE)
            sub_ip1_alt = make_submap(all_batches[i+1], [1], VOXEL_SIZE)
            if sub_i_alt.has_points() and sub_ip1_alt.has_points():
                T_init = all_poses[i][-2] @ np.linalg.inv(all_poses[i+1][1])
                ok2, T2, Info2, fit2, rmse2 = run_icp_multiscale_with_info(
                    source=sub_ip1_alt, target=sub_i_alt, init_transform=T_init
                )
                print(f"    extra {i}->{i+1} fitness={fit2:.3f}, rmse={rmse2:.3f}")
                if ok2:
                    add_edge_with_downweight(pose_graph, i, i+1, T2, Info2, fit2)

    # 4) Loop closure (last -> first)
    print("\n--- Step 4: Loop closure (last -> first) ---")
    last = num_nodes - 1
    sub_last  = make_submap(all_batches[last], [max(0, len(all_batches[last])-2), len(all_batches[last])-1], VOXEL_SIZE)
    sub_first = make_submap(all_batches[0],    [0, min(1, len(all_batches[0])-1)], VOXEL_SIZE)

    loop_added = False
    if sub_last.has_points() and sub_first.has_points():
        T_w_first = all_poses[0][0]
        T_w_last  = all_poses[last][-1]
        T_last_to_first_init = T_w_first @ np.linalg.inv(T_w_last)

        T_coarse, coarse_fit = coarse_global_reg(sub_last, sub_first, VOXEL_SIZE*2.0)
        if T_coarse is not None and coarse_fit > 0.05:
            print("    [Info] Using coarse global registration as loop init.")
            T_init = T_coarse
        else:
            T_init = T_last_to_first_init

        okL, T_last_to_first, InfoL, fitL, rmseL = run_icp_multiscale_with_info(
            source=sub_last, target=sub_first, init_transform=T_init
        )
        print(f"    loop fitness={fitL:.3f}, rmse={rmseL:.3f}")
        if okL:
            add_edge_with_downweight(pose_graph, last, 0, T_last_to_first, InfoL, fitL)
            loop_added = True
        else:
            print("    [Warn] Loop edge weak; will add a soft prior instead.")
    else:
        print("    [Warn] Loop submaps empty; adding a soft prior instead.")

    if not loop_added:
        T_soft = all_poses[0][0] @ np.linalg.inv(all_poses[last][-1])
        Info_soft = np.eye(6) * 0.1  # weak prior
        pose_graph.edges.append(
            o3d.pipelines.registration.PoseGraphEdge(last, 0, T_soft, Info_soft, uncertain=True)
        )
        print("    [Info] Soft last->first prior added.")

    # 5) Optimize
    print("\n--- Step 5: Pose-graph optimization ---")
    option = o3d.pipelines.registration.GlobalOptimizationOption(
        max_correspondence_distance=ICP_DISTANCE_THRESH,
        edge_prune_threshold=0.25,
        reference_node=0
    )
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option
    )

    # 6) Fuse map
    print("\n--- Step 6: Combine map with optimized poses ---")
    pcd_combined = o3d.geometry.PointCloud()
    for i, batch in enumerate(all_batches):
        T_w_i_opt = np.linalg.inv(pose_graph.nodes[i].pose)
        # Skip first (overlap) frame for batches > 0 to avoid duplicates
        start_idx = 1 if i > 0 else 0
        for p in batch[start_idx:]:
            if not p.has_points():
                continue
            q = copy.deepcopy(p)
            q.transform(T_w_i_opt)
            pcd_combined += q

    if APPLY_AXIS_FLIP:
        pcd_combined.transform(AXIS_FLIP)

    # 7) Visualize & (optionally) save
    print("\n--- Step 7: Visualization & Save ---")
    final_pcd = pcd_combined.voxel_down_sample(voxel_size=VOXEL_SIZE)
    print(f"Final combined point cloud has {len(final_pcd.points)} points.")
    o3d.visualization.draw_geometries([final_pcd])

    # Uncomment to write to disk
    # print(f"Saving combined PLY to {output_filename} ...")
    # o3d.io.write_point_cloud(output_filename, final_pcd)
    # print("Done.")

if __name__ == "__main__":
    main()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading MapAnything model...
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main



--- Step 1: Processing all batches ---
Applying distance filter: <= 20.0 m
  Inferring batch of 5 images...
  Inferring batch of 5 images...
  Inferring batch of 5 images...
  Inferring batch of 5 images...

--- Step 2: Build initial poses & nodes ---

--- Step 3: Sequential constraints for neighbors ---
  0->1 fitness=0.765, rmse=0.078
    extra 0->1 fitness=0.677, rmse=0.077
  1->2 fitness=0.448, rmse=0.078
    extra 1->2 fitness=0.441, rmse=0.078
  2->3 fitness=0.248, rmse=0.091
    extra 2->3 fitness=0.176, rmse=0.087
    [Info] Downweighting edge 2->3 (fitness=0.176).

--- Step 4: Loop closure (last -> first) ---
  [Warn] ICP low quality (fitness=0.000, rmse=0.000).
    loop fitness=0.000, rmse=0.000
    [Warn] Loop edge weak; will add a soft prior instead.
    [Info] Soft last->first prior added.

--- Step 5: Pose-graph optimization ---

--- Step 6: Combine map with optimized poses ---

--- Step 7: Visualization & Save ---
Final combined point cloud has 325543 points.
